# Lesson 16 · Determinants and inverses — can this transformation be undone? The determinant decides; the inverse restores

> **How to read this**: the main path first nails down the **2×2** det / inverse formulas; the 3×3 material and extra worked examples in the second half are extra practice — if 2×2 isn't solid yet, don't rush ahead.

**Learning goals**
1. Understand the geometric meaning of the determinant: `det(A)` equals the signed area of the parallelogram spanned by the matrix's column vectors, and `det=0` means space has been squashed flat
2. Implement `det_2x2(M)` and `inv_2x2(M)` with the formula `ad − bc`, returning `None` when `det=0`
3. Compute a 3×3 determinant by cofactor expansion, and check `|A| = 2` against the extra worked example
4. Find a 3×3 inverse with the adjugate formula `A⁻¹ = adj(A)/|A|`, and check `A @ A⁻¹ = I`
5. Understand how `det = 0` affects Aurora's covariance-whitening step (a singular matrix can't be whitened; you need to add a regularization term `ε·I`)

**Why it matters for Aurora**: in Aurora's feature extraction, `det(M) = 0` means the covariance matrix is singular and the whitening step can't go ahead; `A⁻¹` appears directly in the Mahalanobis distance and in least-squares solving.

← **Previous lesson** · [L15 · Gaussian elimination](L15_linear_systems.ipynb)

> Last lesson covered **Gaussian elimination**: the elimination process for a system Ax=b, row-echelon form, and classifying when solutions exist.
> This lesson explores **determinants and inverses**.

## This lesson's story: is the matrix "destroying" information?

Picture a camera. It squeezes three-dimensional space into a two-dimensional photo — the depth information disappears. This transformation is **not invertible**: you can't rebuild the 3D scene from the photo.

Mathematically, this corresponds to `det(A) = 0`: a zero determinant → the matrix has "squashed" space down by one dimension → information is lost → not invertible.

Conversely, `det(A) ≠ 0` → the transformation keeps every dimension → an inverse transformation `A⁻¹` exists.

The geometric meaning of the determinant:
- **2D**: the **signed area** of the unit square after A transforms it (it can be negative, meaning a flip)
- **3D**: the **signed volume** of the unit cube after A transforms it
- **|det| = 2**: the area doubles; **det = 0**: squashed into a line (not invertible); **det < 0**: flipped

In this lesson you'll implement `det_2x2(M)` and `inv_2x2(M)` — the 2×2 case has closed-form formulas you can check by hand.

## 1. The 2×2 determinant: `det = ad − bc`

$$A=\begin{pmatrix}a&b\\c&d\end{pmatrix},\quad \det(A)=ad-bc$$

**Geometric intuition**: imagine [1,0] being sent to [a,c] and [0,1] being sent to [b,d]. The area of the parallelogram these two vectors span is |det(A)|.

**A worked example by hand**: A = [[3, 1], [2, 4]]
`det = 3×4 − 1×2 = 12 − 2 = 10` (areas grow 10 times)

**The inverse** (exists only when det ≠ 0):
$$A^{-1} = \frac{1}{ad-bc}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$$

Notice the pattern: swap the main diagonal, flip the signs on the off-diagonal, divide everything by det.

**A singular matrix** (det = 0): A = [[1, 2], [2, 4]], det = 1×4 − 2×2 = 0.
The two rows are proportional → space is squashed into a line → no inverse exists.

## 1.5 Pause: what does each symbol in `det=ad-bc` correspond to in the picture?

**A student question, revisited**: Ruth and Hannah both asked — what does "span" mean? And why does the area formula for a parallelogram look like `ad-bc`?

Let's start with an everyday analogy. Imagine pinning one end of two rubber bands to a wall, both at the same point (the origin). You stretch the first rubber band to the point `(a, c)`, and the second to the point `(b, d)`. The parallelogram these two rubber bands "stretch open" (a "slanted rectangle") — its area is what the determinant computes.

The two **column vectors** (not row vectors!) of the matrix `A = [[a, b], [c, d]]` are exactly `(a, c)` and `(b, d)` — because by NumPy/linear-algebra convention the matrix's first column is `[a, c]` and its second column is `[b, d]`. The parallelogram these two vectors "span" has area `|ad-bc|`.

**Two concrete examples first, to build the picture**:
- `A = [[2, 0], [0, 2]]`: the two column vectors are `(2, 0)` and `(0, 2)`, two edges stretched to length 2 along the x-axis and the y-axis — that's a square with side 2, area = 4. Check with the formula: `ad-bc = 2×2 - 0×0 = 4`. It matches.
- `A = [[1, 1], [0, 1]]`: the column vectors are `(1, 0)` and `(1, 1)`. The first edge runs along the x-axis with length 1, the second points diagonally to `(1,1)`. This is a parallelogram that's been "pushed over" (a shear transformation), but its base is still 1 and its height is still 1 (the slanted edge only shifts in the x direction; nothing changes in y), so the area is still 1×1=1. Check with the formula: `ad-bc = 1×1 - 1×0 = 1`.

**That's where the formula `ad-bc` comes from — it isn't an arbitrary rule, it's the "base × height" formula worked out in coordinates**: take `(a,c)` as the base; the height of the parallelogram is the length of `(b,d)`'s projection onto the direction perpendicular to `(a,c)`. Work that geometric calculation out and simplify it all the way, and you get `ad-bc` — so at heart the formula is still "base times height," just written in algebra.

The code below draws both examples, so you can see directly what "the spanned parallelogram" looks like.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_span(A, title):
    A = np.array(A, dtype=float)
    col1, col2 = A[:, 0], A[:, 1]
    origin = np.array([0, 0])
    # The parallelogram's four corners: origin -> col 1 -> col 1 + col 2 -> col 2 -> back to the origin
    pts = np.array([origin, col1, col1 + col2, col2, origin])
    plt.plot(pts[:, 0], pts[:, 1], 'o-', color='steelblue')
    plt.fill(pts[:-1, 0], pts[:-1, 1], alpha=0.25, color='steelblue')
    plt.quiver(0, 0, col1[0], col1[1], angles='xy', scale_units='xy', scale=1, color='crimson')
    plt.quiver(0, 0, col2[0], col2[1], angles='xy', scale_units='xy', scale=1, color='seagreen')
    plt.title(f'{title}\n|det|={abs(np.linalg.det(A)):.2f}', fontsize=10)
    plt.axhline(0, color='gray', lw=0.5); plt.axvline(0, color='gray', lw=0.5)
    plt.gca().set_aspect('equal')

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plot_span([[2, 0], [0, 2]], 'square [[2,0],[0,2]]  (red = col 1, green = col 2)')
plt.subplot(1, 2, 2)
plot_span([[1, 1], [0, 1]], 'shear [[1,1],[0,1]]')
plt.tight_layout()
plt.show()
print('Red arrow = first column (a,c), green arrow = second column (b,d); the area of the parallelogram they enclose is |det|.')


## 1.6 How is the inverse formula `[[d,-b],[-c,a]]/det` "solved" for?

Samuel, Ruth and Hannah all asked the same question: is this "swap the main diagonal, flip the signs on the off-diagonal" formula just a rule someone made up, or can it be derived?

The answer: it can be derived, and the derivation uses only something you already know — **solving a system of equations**.

**First, be clear about what condition an "inverse" must satisfy.** By definition, `A⁻¹` is the matrix that, multiplied with `A`, gives back the identity matrix `I=[[1,0],[0,1]]`. Call the four unknowns in `A⁻¹` `x, y, z, w`:

$$A \cdot A^{-1} = \begin{pmatrix}a&b\\c&d\end{pmatrix}\begin{pmatrix}x&y\\z&w\end{pmatrix} = \begin{pmatrix}1&0\\0&1\end{pmatrix}$$

**Expand the matrix product on the left by definition** (each position = the matching row times the matching column, summed term by term):

$$\begin{pmatrix}ax+bz & ay+bw\\cx+dz & cy+dw\end{pmatrix} = \begin{pmatrix}1&0\\0&1\end{pmatrix}$$

That gives us 4 equations — exactly enough to solve for 4 unknowns:
- First column (matching the first column `[1,0]` on the right-hand side): `ax+bz=1`, `cx+dz=0`
- Second column (matching the second column `[0,1]` on the right-hand side): `ay+bw=0`, `cy+dw=1`

**Solve the first column** (`x, z`): from `cx+dz=0` we get `z = -cx/d` (assuming d≠0); substitute back into the first equation to eliminate `z`, tidy up, and solve:
$$x=\frac{d}{ad-bc}, \qquad z=\frac{-c}{ad-bc}$$

**Solve the second column** (`y, w`) with the same elimination, and you get:
$$y=\frac{-b}{ad-bc}, \qquad w=\frac{a}{ad-bc}$$

**Put the four solutions back into a matrix**:
$$A^{-1}=\begin{pmatrix}x&y\\z&w\end{pmatrix}=\frac{1}{ad-bc}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$$

It matches! This is exactly the closed-form formula the course gave — it isn't a rule, it's the result of "solving a system of 4 linear equations." `d` and `a` swapping places (the main diagonal swaps) and `b` and `c` picking up minus signs (the off-diagonal flips sign) both fall out naturally from the elimination — not a mnemonic you have to memorize.

The code below solves both sets of equations for you (with concrete numbers plugged in), so you can see that the `x,y,z,w` it finds are exactly the values the formula gives.

In [ ]:
import numpy as np

# Check the derivation above with concrete numbers: A=[[3,1],[2,4]]
a, b, c, d = 3., 1., 2., 4.
det = a*d - b*c

# Solve the first-column system: a*x+b*z=1, c*x+d*z=0
coef = np.array([[a, b], [c, d]])
x, z = np.linalg.solve(coef, [1, 0])

# Solve the second-column system: a*y+b*w=0, c*y+d*w=1
y, w = np.linalg.solve(coef, [0, 1])

A_inv_solved = np.array([[x, y], [z, w]])
A_inv_formula = (1/det) * np.array([[d, -b], [-c, a]])

print('from solving the systems:\n', np.round(A_inv_solved, 6))
print('from the closed-form formula:\n', np.round(A_inv_formula, 6))
print('do they match:', np.allclose(A_inv_solved, A_inv_formula))


## Notation first: look at the shapes, then the operations

This section works with a square matrix `M` (shape `(n, n)`). The 2×2 determinant is `ad−bc` and can be computed directly; the 3×3 determinant is handled by cofactor expansion, which breaks the problem into three 2×2 minors.

## 2. ✏️ Implement `det_2x2(M)` and `inv_2x2(M)`

**Line of reasoning**:
1. Unpack the matrix: `a, b = M[0]`, `c, d = M[1]`, giving four scalars
2. Compute the determinant: `det = a*d - b*c`; this is the signed area of the parallelogram spanned by the two column vectors `[a,c]` and `[b,d]`
3. Check invertibility: when `det == 0` the matrix squashes the plane into a line and can't be inverted, so return `None` instead of dividing by zero
4. Build the inverse: swap the diagonal entries (`d` moves to the top left, `a` to the bottom right), negate the off-diagonal entries, and multiply everything by `1/det`

**Reference input and output**: `M=[[1,0],[0,2]]` → `det=2`, `inv=[[1,0],[0,0.5]]`; `M=[[1,2],[2,4]]` → `det=0`, not invertible

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Before writing code, fill in the variable table

Three things to pin down before writing `det_2x2`:
- Input: `M`, shape `(2, 2)`, with elements unpacked as `a, b, c, d`
- Key step: `det = a*d - b*c`; when det is 0 there's no inverse
- Returns: `det_2x2` returns a scalar; `inv_2x2` returns a matrix of shape `(2, 2)`

In [ ]:
import numpy as np
def det_2x2(M):
    raise NotImplementedError("To implement: unpack a,b,c,d, then return a*d - b*c")

def inv_2x2(M):
    raise NotImplementedError("To implement: return None when det=0, otherwise return the 2×2 inverse")


## Hands-on: det values and invertibility

Run the cell below and watch how `det` changes with the matrix's entries: when the two rows are proportional, det becomes 0, and `inv_2x2` should refuse to invert.

In [ ]:
import numpy as np

# Determinant = the area scale factor of the unit square after the transformation
mats = {
    'scale 2x':    np.array([[2.,0.],[0.,2.]]),
    'rotate 45°':  np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                           [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'shear':       np.array([[1.,2.],[0.,1.]]),
    'rank-deficient':     np.array([[1.,2.],[2.,4.]]),
}
for name, A in mats.items():
    d = np.linalg.det(A)
    print(f'{name:10s}  det={d:+.4f}')
print('rotation: |det|=1 (area unchanged); rank-deficient: det=0 (squashed to a lower dimension)')


### What exactly is a flip? — Mirror vs. turntable

Samuel and Hannah both got stuck on the same word: the course says "det<0 means a flip (a left hand becomes a right hand)," but the rotation matrix above clearly turns vectors to other directions too — so why is its det positive (`|det|=1`)?

First, an analogy to make the difference between "flipping" and "rotating" clear: imagine a coin in front of you. **Rotating** is spinning the coin around on the table — heads stays up, it just faces a different way. **Flipping** is turning the coin "over" — heads becomes tails, like a mirror swapping left and right (or top and bottom). In the 2D plane, a "flip" corresponds to a **mirror reflection**: folding the whole plane over along some line.

A concrete example: the matrix `F = [[1, 0], [0, -1]]` is a reflection across the x-axis. It sends the point `(x, y)` to `(x, -y)` — the y coordinate changes sign, flipped top to bottom just like a mirror. Compute its determinant: `det(F) = 1×(-1) - 0×0 = -1`, negative as expected.

Compare with a rotation matrix (like the "rotate 45°" above): its det is always 1 (however far it turns), because a rotation only changes direction, doesn't change area, and never needs to "lift the shape up and flip it over."

The plot below compares them: the same asymmetric "L" shape after a rotation and after a reflection (with a symmetric shape you couldn't tell a flip from a rotation, which is why we deliberately chose an asymmetric one). You'll see that the rotated "L" is still a "normal" L (just turned, and it can be turned back by rotation alone); the reflected "L" has become its mirror image (no amount of turning will bring it back) — that's a "flip."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use an asymmetric "L" shape so a flip can be told apart from a rotation (a symmetric shape looks the same either way)
L_shape = np.array([[0,0],[0,2],[0.6,2],[0.6,0.6],[1.4,0.6],[1.4,0],[0,0]]).T  # shape (2, N)

theta = np.pi/4
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])  # rotate 45°, det=+1
F = np.array([[1., 0.], [0., -1.]])  # reflect across the x-axis, det=-1

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, M, title in zip(
    axes,
    [np.eye(2), R, F],
    ['original L', f'rotated 45° (det={np.linalg.det(R):+.1f})', f'reflected (det={np.linalg.det(F):+.1f})']
):
    pts = M @ L_shape
    ax.fill(pts[0], pts[1], alpha=0.4, color='steelblue')
    ax.plot(pts[0], pts[1], color='steelblue')
    ax.set_title(title, fontsize=10)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal'); ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()
print('The rotated L is still a "normal" L (pure turning brings it back to the original); the reflected L has become its mirror image and cannot be turned back — that is the "flip" of det<0.')


## Code experiment: a 3×3 determinant — cofactor expansion by hand vs NumPy

Below we expand the det of a 3×3 matrix along its first row by hand, compare it with NumPy, and check whether the matrix is invertible.

In [ ]:
import numpy as np

# 3x3 determinant by hand vs NumPy
A = np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,10.]])
det_np = np.linalg.det(A)
# Expand along the first row (cofactor expansion)
d0 = A[0,0]*(A[1,1]*A[2,2]-A[1,2]*A[2,1])
d1 =-A[0,1]*(A[1,0]*A[2,2]-A[1,2]*A[2,0])
d2 = A[0,2]*(A[1,0]*A[2,1]-A[1,1]*A[2,0])
det_manual = d0+d1+d2
print(f'NumPy det          = {det_np:.6f}')
print(f'expansion by hand  = {det_manual:.6f}')
print(f'error = {abs(det_np-det_manual):.2e}')
print(f'invertible? {"yes" if abs(det_np)>1e-10 else "no, singular matrix"}')


In [ ]:
M = np.array([[4.,3.],[6.,3.]])
try:
    got_det = det_2x2(M)
    ref_det = np.linalg.det(M)
    print(f"det_2x2 = {got_det:.6f}  numpy = {ref_det:.6f}")
    assert abs(got_det - ref_det) < 1e-9, f"det_2x2 error {abs(got_det - ref_det):.2e}"
    print("✅ determinant passed")
except (NotImplementedError, TypeError) as e:
    print(f"⏭️ det_2x2 not implemented yet: {e}")


In [ ]:
try:
    got_inv = inv_2x2(M)
    ref_inv = np.linalg.inv(M)
    print("inv_2x2:\n", np.round(got_inv, 6))
    print("numpy:\n", np.round(ref_inv, 6))
    assert np.allclose(got_inv, ref_inv), "inverse error out of tolerance"
    print("✅ inverse passed")
except (NotImplementedError, TypeError) as e:
    print(f"⏭️ inv_2x2 not implemented yet: {e}")


### How do you know what kind of "singular matrix" to build for a test?

Samuel asked a very practical question: the test case below uses `singular = [[1,2],[2,4]]`, but for a different problem, how would you think of building a det=0 example like that? Is there some kind of "recipe"?

There is a simple, reliable recipe: **just make one row of the matrix a multiple of another** (the same works for columns), and its det is guaranteed to be 0.

Why? Go back to the "spanned parallelogram" picture (section 1.5): if the second vector `(b,d)` is just the first vector `(a,c)` times some multiple `k` (that is, `b=ka, d=kc`), then the two vectors point in exactly the same (or opposite) direction, and they can't "span" a parallelogram with any area — they can only lie on the same line, so the area is naturally 0. Check with the formula: `det = a(kc) - (ka)c = kac - kac = 0`, always.

**The recipe**: pick any vector `v = (p, q)` and build `A = [[p, q], [k*p, k*q]]` (row 2 is k times row 1); this matrix is guaranteed to be singular. For example, `v=(1,2)`, `k=2` gives the example used in the test below, `[[1,2],[2,4]]`.

The code below generates a few "proportional matrices" at random and checks that their det really is 0:

In [ ]:
import numpy as np

for p, q, k in [(1, 2, 2), (3, -1, 4), (0.5, 5, -2)]:
    A_sing = np.array([[p, q], [k*p, k*q]])
    print(f'v=({p},{q}), k={k}  ->  A={A_sing.tolist()}  det={np.linalg.det(A_sing):.6f}')
print('However v and k are chosen, as long as row 2 is a multiple of row 1, det is always 0.')


In [ ]:
# Singular-matrix asserts — checking the core branch the spec promises
# singular = [[1,2],[2,4]], det = 1*4-2*2 = 0
try:
    singular = np.array([[1., 2.], [2., 4.]])
    assert det_2x2(singular) == 0.0, f"the determinant of the singular matrix should be 0, got {det_2x2(singular)}"
    assert inv_2x2(singular) is None, f"a singular matrix should return None, got {inv_2x2(singular)}"
    print("✅ singular matrix passed")
except (NotImplementedError, TypeError) as e:
    print(f"⏭️ functions not implemented yet, skipping the singular-matrix test: {e}")


## 3. The 3×3 determinant: **cofactor expansion** along the first row (extra worked example)

Extra worked example: $A=\begin{pmatrix}1&1&0\\1&2&1\\0&1&3\end{pmatrix}$, answer $|A|=2$.

$|A| = a_{11}M_{11} - a_{12}M_{12} + a_{13}M_{13}$, where $M_{ij}$ is the 2×2 determinant left after crossing out row i and column j (the minor).


### Reading the NumPy indexing in the code: what does `A[1:, [1,2]]` mean?

Hannah got stuck here: she couldn't read the `A[1:,[1,2]]` in the code below. Let's take it apart. Inside NumPy's square brackets, what comes before and after the comma controls different dimensions: **before the comma picks rows, after the comma picks columns**.

- `A[1:, ...]`: `1:` means "slice from row 1 to the end," i.e. **skip row 0** (Python counts from 0). This step does the "crossing out row 0."
- `A[..., [1,2]]`: the `[1,2]` inside the brackets is a list, meaning "only columns 1 and 2" (again counting from 0), skipping column 0. This is called **fancy indexing** — using a list to say which positions you want, and they don't need to be consecutive.

Put together, `A[1:, [1,2]]` = "drop row 0, keep only columns 1 and 2" — that is, row 0 and column 0 are crossed out, and the 2×2 submatrix left over is exactly the block for the minor `M11` (cross out row 1, column 1, in 1-indexed notation here).

Likewise:
- `M12` crosses out row 0 and column 1, so it keeps columns 0 and 2: `A[1:, [0,2]]`
- `M13` crosses out row 0 and column 2, so it keeps columns 0 and 1: `A[1:, [0,1]]`

**A small tip**: we can use the simple slice `1:` here only because we always cross out row 0. To cross out an arbitrary row `i` and column `j`, slicing gets awkward — later we'll use the utility function `np.ix_(rows, cols)`, which does the same thing but works for any `i, j` (you'll see it shortly).

In [ ]:
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
# Expand along the first row
M11 = np.linalg.det(A[1:,[1,2]])   # cross out row 0, column 0
M12 = np.linalg.det(A[1:,[0,2]])   # cross out row 0, column 1
M13 = np.linalg.det(A[1:,[0,1]])   # cross out row 0, column 2
detA = A[0,0]*M11 - A[0,1]*M12 + A[0,2]*M13
print('|A| by cofactor expansion =', round(detA), ' (course notes: 2)')

### Does the answer change if you expand along a different row?

Samuel raised a good question: the course only showed "expanding along the first row" — if you expand along the second or third row instead, will the answer still be `|A|=2`?

The answer: **expanding along any row (or any column) gives exactly the same result** — that's a mathematical property of the determinant. An intuitive way to see it: the determinant computes one and the same "volume scaling factor," so whichever direction you choose to break the calculation down from, you should be measuring the same volume of the same space — measuring from a different angle shouldn't change the result.

**The one thing to watch is the sign pattern**: when expanding along row `i` (or column `j`), the sign in front of each term is `(-1)^(i+j)` (counting `i, j` from 1 here). Expanding along the first row gives `+, -, +` (`i=1`, `j=1,2,3` → `(-1)^2, (-1)^3, (-1)^4` = `+,-,+`); expanding along the second row, the signs become `-, +, -` (`i=2` → `(-1)^3,(-1)^4,(-1)^5` = `-,+,-`).

The code below expands the same matrix along the second and third rows and checks that both answers are still 2.

In [ ]:
import numpy as np

A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)

def minor(A, i, j):
    """Determinant of the 2x2 submatrix left after crossing out row i and column j (i, j counted from 0)"""
    rows = [r for r in range(3) if r != i]
    cols = [c for c in range(3) if c != j]
    sub = A[np.ix_(rows, cols)]
    return sub[0, 0]*sub[1, 1] - sub[0, 1]*sub[1, 0]

# Expand along row 2 (0-indexed i=1): signs -, +, -
det_row2 = -A[1,0]*minor(A,1,0) + A[1,1]*minor(A,1,1) - A[1,2]*minor(A,1,2)
# Expand along row 3 (0-indexed i=2): signs +, -, +
det_row3 =  A[2,0]*minor(A,2,0) - A[2,1]*minor(A,2,1) + A[2,2]*minor(A,2,2)

print('expanding along row 1 (already done in the course) = 2')
print(f'expanding along row 2                          = {det_row2:.6f}')
print(f'expanding along row 3                          = {det_row3:.6f}')
print(f'NumPy reference value                          = {np.linalg.det(A):.6f}')
print('All three expansions agree — whichever row you choose does not affect the answer; the row with the most 0s usually takes the least work.')


### Putting the pieces together: minor → cofactor → adjugate

All three students got stuck here: minor, cofactor, adjugate — the words sound alike, so which is which? Let's build them up layer by layer, "from small to big."

**Layer 1: the minor (you can already compute it)** — `Minor(i,j)`, the course's `M_ij`: the determinant of the smaller matrix left after crossing out row `i` and column `j`. You've just practiced it; `M11`, for example, is the 2×2 determinant left after crossing out row 1 and column 1.

**Layer 2: the cofactor (one extra sign)** — `Cofactor(i,j) = (-1)^(i+j) × Minor(i,j)`. This `(-1)^(i+j)` is the "checkerboard sign" mentioned earlier: mark positions where `i+j` is even with `+` and odd ones with `-`, and laid out they alternate just like a checkerboard:

```
+ - +
- + -
+ - +
```

**A small clarification about signs** (answering Hannah's question): the course's line `|A| = a11*M11 - a12*M12 + a13*M13` uses "bare" minors (with no built-in sign), and the `+,-,+` is written into the formula by hand; if you use cofactors instead (which already carry the sign), the formula becomes `|A| = a11*C11 + a12*C12 + a13*C13`, with no plus or minus signs at all — the two ways of writing it are really the same thing; the only difference is whether the sign is "put in the formula" or "put into the definition of C."

**Layer 3: the adjugate (put all the cofactors together, then transpose)** — first compute the cofactor at every position `(i,j)` of the matrix and arrange them into a "cofactor matrix"; then **transpose** it (swap rows and columns), and only that is the adjugate `adj(A)`. Why transpose at the end? Because the inverse formula `A⁻¹ = adj(A)/|A|` requires this arrangement — expand `A · adj(A) = |A|·I` to check, and you'll find the identity fails without the transpose. This step is easy to overlook, and it's where 3×3 inversion most often goes wrong.

**Now let's compute all three layers on the course's example**, still the matrix `A = [[1,1,0],[1,2,1],[0,1,3]]`:

In [ ]:
import numpy as np

A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)

def cofactor(A, i, j):
    return (-1)**(i+j) * minor(A, i, j)

# Layers 1 + 2: compute all 9 cofactors and arrange them into the "cofactor matrix"
C = np.array([[cofactor(A, i, j) for j in range(3)] for i in range(3)])
print('cofactor matrix C (not yet transposed):\n', C)

# Layer 3: transpose to get the adjugate
adj = C.T
print('\nadjugate adj(A) = transpose of C:\n', adj)

detA = np.linalg.det(A)
A_inv = adj / detA
print(f'\ndet(A) = {detA}')
print('A^-1 = adj(A)/det(A):\n', A_inv)
print('\nmatches the course answer 0.5*[[5,-3,1],[-3,3,-1],[1,-1,1]]:',
      np.allclose(A_inv, 0.5*np.array([[5,-3,1],[-3,3,-1],[1,-1,1]])))


## 4. Inverting with the adjugate: `A⁻¹ = adj(A) / |A|` (extra worked example)

Result of the extra worked example (A is symmetric):
$$A^{-1}=\tfrac12\begin{pmatrix}5&-3&1\\-3&3&-1\\1&-1&1\end{pmatrix}$$


In [ ]:
inv_expected = 0.5*np.array([[5,-3,1],[-3,3,-1],[1,-1,1]], float)
print('numpy inverse matches the course notes:', np.allclose(np.linalg.inv(A), inv_expected))
print('check A · A⁻¹ = I:\n', np.round(A @ inv_expected, 6))
assert np.allclose(A @ inv_expected, np.eye(3))
print('✅ adjugate inverse matches the course notes.')

## ✏️ Exercise: implement `det_3x3(M)` and `inv_3x3(M)`

Learning goals 3 and 4 ask you to implement the 3×3 determinant and inverse by hand; the extra worked example above (`A=[[1,1,0],[1,2,1],[0,1,3]]`, `|A|=2`) is the test case.

### Before writing `inv_3x3`, make a checklist

Ruth pointed out that, having read this far, writing 9 cofactors in one go while minding the transpose and the division makes it easy to slip up halfway. Here's a step-by-step checklist you can follow, breaking the big problem into small ones:

1. **Write `det_3x3(M)`**: reuse the earlier first-row expansion — compute the three minors with `minor(M,0,j)`, pair them with the signs `+,-,+` (or use `cofactor(M,0,j)` directly and add all three terms), and sum the three terms.
2. **Write `cofactor(M,i,j)`** (if you don't have it yet): `(-1)**(i+j) * minor(M,i,j)`.
3. **Assemble the cofactor matrix**: with two nested loops, compute `cofactor(M,i,j)` for every `(i,j)` (`i,j` each taking 0,1,2) and fill it into a new 3×3 matrix.
4. **Transpose**: `.T` of the cofactor matrix is the adjugate `adj(M)` — don't forget this step; it's the one most easily missed.
5. **Divide by the determinant**: `inv_3x3(M) = adj(M) / det_3x3(M)`; don't forget to check the `det_3x3(M) == 0` case first, where you should return `None`.

The code in the "Putting the pieces together" section above has already demonstrated steps 1–4 (using the custom `minor`/`cofactor` helper functions) — when writing `inv_3x3` you can reuse the same structure, just plugging in your own `det_3x3` at the end (rather than `np.linalg.det`).

In [ ]:
def det_3x3(M):
    # ✏️ TODO: cofactor expansion along the first row
    # Hint: compute each minor with det_2x2 (once det_2x2 is implemented, use it in place of np.linalg.det)
    raise NotImplementedError("Implement the 3×3 determinant by cofactor expansion")

def inv_3x3(M):
    # ✏️ TODO: implement with the adjugate, adj(M) / det_3x3(M)
    raise NotImplementedError("Implement the 3×3 inverse with the adjugate formula")


In [ ]:
# Check the 3×3 implementation
A3 = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
inv_expected_3x3 = 0.5 * np.array([[5,-3,1],[-3,3,-1],[1,-1,1]], float)
try:
    assert abs(det_3x3(A3) - 2.0) < 1e-9, f"det_3x3 error: {det_3x3(A3)}"
    assert np.allclose(inv_3x3(A3), inv_expected_3x3), "inv_3x3 error out of tolerance"
    print("✅ 3×3 determinant and inverse passed")
except (NotImplementedError, TypeError) as e:
    print(f"⏭️ 3×3 functions not implemented yet: {e}")


**🔗 Aurora connection**: the covariance matrix `Σ` must be invertible in the whitening step of Aurora's feature extraction (this feature isn't implemented yet; there's no features.py under `src/aurora/audio/` at the moment, and the implementation should add a regularization term `ε·I` first to guarantee invertibility); `det(Σ) = 0` means the energies of some frequency bands are perfectly correlated, so the whitening matrix `Σ^(-1/2)` can't be computed, and you need to add the regularization term `ε·I` first. The Mahalanobis distance `d² = (x−μ)ᵀ Σ⁻¹ (x−μ)` depends directly on `Σ⁻¹` — a singular covariance makes the distance computation blow up numerically.

**What the extra worked examples cover**: determinant, minor, cofactor, adjugate, inverse.

## 🎨 Picture: the columns of a matrix (determinant = the signed area the columns span)

The `matrix_4ways(A)` below draws "four ways of reading" the same matrix — it is **not** the parallelogram geometry plot drawn with `plot_span` back in section 1.5. The four panels are: ① an overall heat map (color intensity shows the size of each value); ② each cell labeled with its actual value; ③ the matrix highlighted "by columns" (the column vectors that "span" the parallelogram); ④ the matrix highlighted "by rows." If you don't see any image after running it, first make sure the notebook kernel supports inline plotting (Jupyter does by default); you can also just check that no error was raised to confirm the function ran normally.

In [ ]:
from laviz import style, matrix_4ways
style()
matrix_4ways([[1,1,0],[1,2,1],[0,1,3]]);

In [ ]:
import numpy as np

# Parameter experiment: how the determinant changes with the matrix entries
# A = [[a, 1],[1, a]]  => det = a²-1
for a in [-2., -1., 0., 1., 2., 3.]:
    A = np.array([[a, 1.],[1., a]])
    d = np.linalg.det(A)
    inv = 'invertible' if abs(d)>1e-10 else 'singular'
    print(f'a={a:+.0f}  det={d:+.2f}  → {inv}')
print('→ when a=±1, det=0 and the matrix is singular, not invertible.')


## Parameter experiment: random matrices vs. singular matrices

Build `A = np.random.rand(3, 3)` several times and print `np.linalg.det(A)`, confirming that a random 3×3 matrix is almost always invertible. Then set the second row to twice the first, `A[1] = 2 * A[0]`, recompute det, and confirm it becomes 0 (or, because of floating-point rounding, extremely close to 0). Also compare the two cases with `np.linalg.cond(A)`: a random matrix's condition number (κ) is usually somewhere around 10–100, while a singular matrix's condition number shoots up past 1e15.

## Wrap-up

You can now use `det_2x2` and `inv_2x2` to check whether any 2×2 matrix is invertible, and use cofactor expansion for the 3×3 case. In Aurora's covariance-whitening step, compute det first to decide whether it's safe to call the inverse. The next lesson's eigendecomposition uses det=0 to recognize singular matrices — a direct extension of today's conclusion.

---
⬇️ **Checkpoint**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: determinants and inverses by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: A = [[3, 1], [2, 4]] — compute det(A) by hand. Formula: ad − bc.

**Question 2**: B = [[4, 3], [6, 3]] — compute det(B) by hand, then write down B⁻¹ with the closed-form formula.
Formula: B⁻¹ = (1/det(B)) · [[d, -b], [-c, a]]

**Question 3**: C = [[1, 2], [2, 4]], det(C) = ? Can you find C⁻¹? Why or why not?

**Question 4**: if det(A) = −5, what does that mean geometrically?
(What's the area scale factor? Was there a flip?)

> Hint: if you're not sure what "flip" means exactly, look back at the reflection-matrix example in the earlier section "What exactly is a flip? — Mirror vs. turntable."

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: det([[3,1],[2,4]])
A1 = np.array([[3., 1.], [2., 4.]])
try:
    d1 = det_2x2(A1)
    assert np.isclose(d1, 10.0, atol=1e-10), f"det should be 10, got {d1}"
    print(f"Q1 ✅  det([[3,1],[2,4]]) = 3×4-1×2 = {d1}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement det_2x2() first, then run the answer-check cell")

# Q2: inv([[4,3],[6,3]])
B = np.array([[4., 3.], [6., 3.]])
det_B = 4*3 - 3*6  # = 12-18 = -6
B_inv_ref = np.linalg.inv(B)
try:
    got_inv = inv_2x2(B)
    assert np.allclose(got_inv, B_inv_ref, atol=1e-10)
    print(f"Q2 ✅  det(B)={det_B}, B⁻¹ = (1/{det_B})·[[3,-3],[-6,4]] = \n{got_inv}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement inv_2x2() first, then run the answer-check cell")
assert np.isclose(det_B, -6.0, atol=1e-10)

# Q3: singular matrix, det=0
C = np.array([[1., 2.], [2., 4.]])
det_C_hand = 1*4 - 2*2
assert det_C_hand == 0
print(f"\nQ3 ✅  det([[1,2],[2,4]]) = {det_C_hand} → singular matrix, not invertible (the two rows are proportional)")

# Q4: geometric meaning of det=-5
print(f"\nQ4 ✅  det(A)=-5:")
print(f"       |det|=5 → areas grow 5 times")
print(f"       det<0   → a flip happened (a left hand becomes a right hand; the signed area is negative)")
print("\n🎉 Determinant and inverse whiteboard challenge passed! det=0 ↔ not invertible is now yours.")

In [ ]:
# ✏️ Self-review for this lesson
l16_review = {
    "det_2x2_implemented":  None,  # det_2x2 implemented and passes its asserts? True/False
    "inv_2x2_implemented":  None,  # inv_2x2 implemented and passes its asserts? True/False
    "det_geometry_intuition": None, # understand det = signed area/volume scaling? True/False
    "singular_det_zero":    None,  # know det=0 ↔ not invertible? True/False
    "whiteboard_passed":    None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l16_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l16_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L16 complete! On to L17: eigendecomposition')

---

→ **Next lesson** · [L17 · Eigendecomposition A = PDP⁻¹](L17_eigen_diagonalization.ipynb)

> Next lesson covers **eigendecomposition A = PDP⁻¹**: changing basis turns a matrix product into scalar multiplication.